# 1 Load Data


Read a CSV file. This file contains all metadata for Task 2, including:
- **Original_id**: The draft resolution's ID.
- **Country (Voting/Representing Country)**: This identifies the nation the large language model (LLM) must represent in Task 2 (the voting simulation). For example, in the first row, the model must assume the role of a diplomat from “Ukraine (UKRAINE)” to decide whether to vote in favor of, against, or abstain on draft resolution 2148.
- **Authors (Drafting/Proposing Countries)**: This refers to the countries that drafted or co-sponsored the resolution draft, typically termed penholders or co-penholders in UN terminology.
- **Voting**: The country's actual voting outcome.



In [2]:
from google.colab import drive
import pandas as pd
import os

# mount the google drive
drive.mount('/content/drive')

# Define the absolute path to your project's root directory
# "MyDrive" means "我的云端硬盘" in Chinese
BASE_DIR = '/content/drive/MyDrive/FTEC5660/Individual project'

csv_path = os.path.join(BASE_DIR, 'data/task2.csv')
df = pd.read_csv(csv_path)

# Test whether the read operation was successful
df.head()


Mounted at /content/drive


,Country,Original_id,Authors,Voting,Date
0,UKRAINE,2148,['France'],Y,2017-01-27
1,POLAND,2218,['United States'],Y,2018-04-13
2,MALTA,2531,"['Afghanistan', 'Albania', 'Algeria', 'Andorra...",Y,2024-05-24
3,POLAND,2268,['United Kingdom'],Y,2019-03-27
4,EGYPT,2152,['United Kingdom'],Y,2017-02-23


# 2 Load Draft Content

Read the draft ID and locate the corresponding JSON text file in data/task2/.

- **Logic**: It iterates through each draft ID, opens the corresponding English JSON file (endswith(‘EN.json’)), and extracts the draft's plain text content (draft[‘Content’]).

- **Final result**: Three core Python lists are prepared: countries (list of nations), drafts (list of draft texts), and votes (list of actual voting results). These three maintain a one-to-one correspondence.

In [4]:
import os
import json

# Read Draft Id
draft_ids = list(df['Original_id'])

# input data
countries = list(df['Country']) # the nation the LLM will represent
drafts = [] # draft content
votes = list(df['Voting']) # groundtruth vote results

# get draft content
draft_content_path = os.path.join(BASE_DIR, 'data/task2')
for i in draft_ids:
    folder_path = os.path.join(draft_content_path, str(i))
    files = os.listdir(folder_path)
    json_file = [file for file in files if file.endswith('EN.json')][0]
    with open(os.path.join(folder_path, json_file)) as f:
        draft = json.load(f)
    drafts.append(draft['Content'])

# 3 Configure Model API

Together API is a platform integrating multiple open-source large language models. To run this code, you need to enter your own API key in the “Secret” field on the left sidebar.

Access Together API key:https://api.together.ai/settings/api-keys

Model List:https://api.together.ai/models

In [6]:
!pip install together

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.8/330.8 kB 21.2 MB/s eta 0:00:00


In [8]:
# If use Together API
from together import Together
from google.colab import userdata

# Save your own API key in the Colab Secret with the same name!
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

# Initialize Client
your_model_name = 'meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo'
your_api_key = TOGETHER_API_KEY
client = Together(api_key=your_api_key)

# 4 Model inference process

Place each draft and country into the prompt, send it to the large model, and collect the model's predictions.

- **Prompt**:
1. system_prompt: Establishes the model's persona and strictly limits output to only ‘Y’ (Yes), ‘N’ (No), or ‘A’ (Abstain).
2. user_prompt: Fills in the specific {country} and {draft} (draft text) into the template.

- **Error Handling Mechanism**: Since LLMs occasionally produce inconsistent responses (outputting non-compliant formats), the author implemented a safeguard: if the model's answer isn't ‘Y’, ‘N’, or ‘A’, it prints an error message and randomly selects a valid response using `random.choice(valid_votes)` to prevent code crashes.

In [9]:
import random
from tqdm import tqdm

pred = []
invalid_responses = []
for i, (draft, country) in tqdm(enumerate(zip(drafts, countries))):
    system_prompt = "You are an experienced diplomat participating in United Nations Security Council sessions. Your task is to read draft resolutions and vote accordingly. You must strictly respond with 'Y', 'N', or 'A' without any additional explanation."
    user_prompt = f"""The following is a United Nations Security Council draft resolution. Assume you are a diplomat from {country} and must cast your vote.

    Please read the draft and vote using one of the following options:
    - 'Y' for Yes (in favor)
    - 'N' for No (against)
    - 'A' for Abstain

    Draft Resolution: {draft}
    Answer:
    """
    response = client.chat.completions.create(
        model=your_model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=1,
        temperature=0.0
    )
    result = response.choices[0].message.content.strip()
    valid_votes = ['Y', 'N', 'A']
    if result not in valid_votes:
        print(f"Invalid response: {result}")
        result = random.choice(valid_votes)
        invalid_responses.append(i)
    pred.append(result)

30it [00:15,  1.92it/s]


# 5 Evaluate

## Calculate_metrics
A function named `calculate_metrics(pred, labels)` is defined to compute the discrepancy between predicted results and actual labels.

- **Logic**: Utilizes the scikit-learn library. First, the alphabetic labels (Y/N/A) are converted to numeric values (0/1/2) using LabelEncoder for easier computational processing.

A comprehensive set of standard metrics for machine learning classification tasks is computed: Accuracy, Macro AUC, Balanced Accuracy, and others. These directly correspond to the data sources in Table 2 of the paper.

In [10]:
# calculate metrics
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
from sklearn.preprocessing import LabelEncoder, label_binarize
from imblearn.metrics import geometric_mean_score
import numpy as np

def calculate_metrics(pred, labels):
    label_encoder = LabelEncoder()
    all_classes = list(set(labels) | set(pred))
    label_encoder.fit(all_classes)

    labels = label_encoder.transform(labels)
    pred = label_encoder.transform(pred)

    acc = accuracy_score(labels, pred)

    num_classes = len(label_encoder.classes_)
    true_labels_bin = label_binarize(labels, classes=list(range(num_classes)))
    pred_bin = label_binarize(pred, classes=list(range(num_classes)))

    auc = roc_auc_score(true_labels_bin, pred_bin, multi_class='ovr', average='macro')
    pr_auc = average_precision_score(true_labels_bin, pred_bin, average='macro')

    balanced_acc = balanced_accuracy_score(labels, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, pred, average='macro')

    mcc = matthews_corrcoef(labels, pred)
    g_mean = geometric_mean_score(labels, pred, average='macro')

    print(f'Accuracy: {acc}')
    print(f'AUC: {auc}')
    print(f'Balanced Accuracy: {balanced_acc}')
    print(f'Precision: {prec}')
    print(f'Recall: {rec}')
    print(f'F1: {f1}')
    print(f'PR AUC: {pr_auc}')
    print(f'MCC: {mcc}')
    print(f'G-Mean: {g_mean}')

    print('Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean')
    print(f'{acc:.4f} {auc:.4f} {balanced_acc:.4f} {prec:.4f} {rec:.4f} {f1:.4f} {pr_auc:.4f} {mcc:.4f} {g_mean:.4f}')


## Excecute

In [15]:
calculate_metrics(pred, votes)

Accuracy: 0.9333333333333333
AUC: 0.5
Balanced Accuracy: 0.5
Precision: 0.4666666666666667
Recall: 0.5
F1: 0.4827586206896552
PR AUC: 0.9333333333333333
MCC: 0.0
G-Mean: 0.5
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean
0.9333 0.5000 0.5000 0.4667 0.5000 0.4828 0.9333 0.0000 0.5000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 6 Modification about prompt

## Add draft authors

Add Author metadata in the prompt.

Author refers to the countries that drafted or co-sponsored the resolution draft, typically termed penholders or co-penholders in UN terminology. It may carry geopolitical implications.

In [12]:
import ast # Used to parse strings like “[‘France’]”

# Extract and clean the Authors column
def clean_authors(author_str):
    try:
        # Convert “[‘France’, ‘United States’]” into a genuine Python list
        author_list = ast.literal_eval(author_str)
        # Connect the country names with commas to form “France, United States”
        return ", ".join(author_list)
    except:
        return str(author_str)

authors = [clean_authors(a) for a in df['Authors']]

print(f"Drafting Country: {authors[0]}")

Drafting Country: France


In [13]:
import random
from tqdm import tqdm

pred2 = []
invalid_responses2 = []

# New: Added authors list to the zip file
for i, (draft, country, author) in tqdm(enumerate(zip(drafts, countries, authors))):

    system_prompt = "You are an experienced diplomat participating in United Nations Security Council sessions. Your task is to read draft resolutions and vote accordingly. You must strictly respond with 'Y', 'N', or 'A' without any additional explanation."

    # New addition: Added author country information to user_prompt
    user_prompt = f"""The following is a United Nations Security Council draft resolution authored/sponsored by: {author}.

    Assume you are a diplomat from {country} and must cast your vote.

    Please read the draft and vote using one of the following options:
    - 'Y' for Yes (in favor)
    - 'N' for No (against)
    - 'A' for Abstain

    Draft Resolution: {draft}
    Answer:
    """

    response = client.chat.completions.create(
        model=your_model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=1,
        temperature=0.0
    )

    result = response.choices[0].message.content.strip()
    valid_votes = ['Y', 'N', 'A']
    if result not in valid_votes:
        print(f"Invalid response: {result}")
        result = random.choice(valid_votes)
        invalid_responses2.append(i)

    pred2.append(result)

30it [00:12,  2.37it/s]


In [14]:
calculate_metrics(pred2, votes)

Accuracy: 0.9333333333333333
AUC: nan
Balanced Accuracy: 0.9642857142857143
Precision: 0.5555555555555555
Recall: 0.6428571428571429
F1: 0.5876543209876544
PR AUC: 0.5539682539682539
MCC: 0.6846836658606624
G-Mean: 0.7925031384731521
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean
0.9333 nan 0.9643 0.5556 0.6429 0.5877 0.5540 0.6847 0.7925


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Add draft dates

Add draft date to the prompt

In [16]:
dates = list(df['Date']) # Extract date data

In [17]:
pred_add_date_only = []
invalid_responses_add_date_only = []

# New: Add dates to the zip file
for i, (draft, country, date) in tqdm(enumerate(zip(drafts, countries, dates))):

    system_prompt = "You are an experienced diplomat participating in United Nations Security Council sessions. Your task is to read draft resolutions and vote accordingly. You must strictly respond with 'Y', 'N', or 'A' without any additional explanation."

    # New: Add only the date (date) to user_prompt
    user_prompt = f"""The following is a United Nations Security Council draft resolution proposed on {date}.

    Assume you are a diplomat from {country} and must cast your vote.

    Please read the draft and vote using one of the following options:
    - 'Y' for Yes (in favor)
    - 'N' for No (against)
    - 'A' for Abstain

    Draft Resolution: {draft}
    Answer:
    """

    response = client.chat.completions.create(
        model=your_model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=1,
        temperature=0.0
    )

    result = response.choices[0].message.content.strip()
    valid_votes = ['Y', 'N', 'A']
    if result not in valid_votes:
        result = random.choice(valid_votes)
        invalid_responses_add_date_only.append(i)

    pred_add_date_only.append(result)


30it [00:13,  2.25it/s]


In [18]:
calculate_metrics(pred_add_date_only, votes)

Accuracy: 0.9
AUC: 0.48214285714285715
Balanced Accuracy: 0.48214285714285715
Precision: 0.46551724137931033
Recall: 0.48214285714285715
F1: 0.47368421052631576
PR AUC: 0.9311165845648605
MCC: -0.04962916669854651
G-Mean: 0.48214285714285715
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean
0.9000 0.4821 0.4821 0.4655 0.4821 0.4737 0.9311 -0.0496 0.4821


## Add both authors and dates

In [19]:
pred_date_and_author = []
invalid_responses_both = []

In [20]:
# New: Include both authors and dates in the zip file
for i, (draft, country, author, date) in tqdm(enumerate(zip(drafts, countries, authors, dates))):

    system_prompt = "You are an experienced diplomat participating in United Nations Security Council sessions. Your task is to read draft resolutions and vote accordingly. You must strictly respond with 'Y', 'N', or 'A' without any additional explanation."

    # New: Add both the drafting country (author) and date (date) to user_prompt.
    user_prompt = f"""The following is a United Nations Security Council draft resolution authored/sponsored by: {author}, and proposed on {date}.

    Assume you are a diplomat from {country} and must cast your vote.

    Please read the draft and vote using one of the following options:
    - 'Y' for Yes (in favor)
    - 'N' for No (against)
    - 'A' for Abstain

    Draft Resolution: {draft}
    Answer:
    """

    response = client.chat.completions.create(
        model=your_model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=1,
        temperature=0.0
    )

    result = response.choices[0].message.content.strip()
    valid_votes = ['Y', 'N', 'A']
    if result not in valid_votes:
        result = random.choice(valid_votes)
        invalid_responses_both.append(i)

    pred_date_and_author.append(result)



30it [00:13,  2.26it/s]


In [21]:
calculate_metrics(pred_date_and_author, votes)

Accuracy: 0.9
AUC: nan
Balanced Accuracy: 0.9464285714285714
Precision: 0.5
Recall: 0.6309523809523809
F1: 0.5366876310272537
PR AUC: 0.4976190476190476
MCC: 0.6000415268021398
G-Mean: 0.7803327003303382
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean
0.9000 nan 0.9464 0.5000 0.6310 0.5367 0.4976 0.6000 0.7803


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 7 Modification about Model

## Llama 4 Maverick Instruct (17Bx128E)

Modify the model to **Llama 4 Maverick Instruct (17Bx128E)** model in Together API.

Details about this model: https://api.together.ai/models/meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8

In [33]:
your_model_name_2 = 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8'

In [34]:
pred_model_2 = []
invalid_responses_model_2 = []
for i, (draft, country) in tqdm(enumerate(zip(drafts, countries))):
    system_prompt = "You are an experienced diplomat participating in United Nations Security Council sessions. Your task is to read draft resolutions and vote accordingly. You must strictly respond with 'Y', 'N', or 'A' without any additional explanation."
    user_prompt = f"""The following is a United Nations Security Council draft resolution. Assume you are a diplomat from {country} and must cast your vote.

    Please read the draft and vote using one of the following options:
    - 'Y' for Yes (in favor)
    - 'N' for No (against)
    - 'A' for Abstain

    Draft Resolution: {draft}
    Answer:
    """
    response = client.chat.completions.create(
        model=your_model_name_2,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=1,
        temperature=0.0
    )
    result = response.choices[0].message.content.strip()
    valid_votes = ['Y', 'N', 'A']
    if result not in valid_votes:
        print(f"Invalid response: {result}")
        result = random.choice(valid_votes)
        invalid_responses_model_2.append(i)
    pred_model_2.append(result)

30it [00:16,  1.81it/s]


In [35]:
calculate_metrics(pred_model_2, votes)

Accuracy: 0.9333333333333333
AUC: nan
Balanced Accuracy: 0.5
Precision: 0.3333333333333333
Recall: 0.3333333333333333
F1: 0.3333333333333333
PR AUC: 0.35555555555555557
MCC: 0.5
G-Mean: 0.5708992257184502
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean
0.9333 nan 0.5000 0.3333 0.3333 0.3333 0.3556 0.5000 0.5709


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-d

## Gemini-2.0-flash

Modify the model to **Gemini 2.0 Flash** model in Vertex API.

In [47]:
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.8 MB/s eta 0:00:00


In [50]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# Set your own key in the Colab Secret
VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    api_key=VERTEX_API_KEY,
    vertexai=True,
    temperature=0.0,
    max_tokens=5
)

In [52]:
import time

pred_gemini = []
invalid_responses_gemini = []

for i, (draft, country) in tqdm(enumerate(zip(drafts, countries))):

    system_content = "You are an experienced diplomat participating in United Nations Security Council sessions. Your task is to read draft resolutions and vote accordingly. You must strictly respond with 'Y', 'N', or 'A' without any additional explanation."

    user_content = f"""The following is a United Nations Security Council draft resolution. Assume you are a diplomat from {country} and must cast your vote.

    Please read the draft and vote using one of the following options:
    - 'Y' for Yes (in favor)
    - 'N' for No (against)
    - 'A' for Abstain

    Draft Resolution: {draft}

    Now, please output your final vote (Y, N, or A) based on your diplomatic stance:"""

    messages = [
        SystemMessage(content=system_content),
        HumanMessage(content=user_content)
    ]

    try:
        response = llm.invoke(messages)
        raw_result = response.content.strip()

        # Smart Extraction Logic: Find the first occurrence of Y, N, or A
        result = ""
        for char in raw_result.upper():
            if char in ['Y', 'N', 'A']:
                result = char
                break

        valid_votes = ['Y', 'N', 'A']
        if result not in valid_votes:
            print(f"\n[Debug] Invalid response at index {i}: '{raw_result}'")
            result = random.choice(valid_votes)
            invalid_responses_gemini.append(i)

        pred_gemini.append(result)

    except Exception as e:
        # Exception Handling: If an API error occurs (such as network disconnections or quota limitations), it will not cause the entire program to crash.
        print(f"\n[Error] API call failed at index {i}. Reason: {e}")
        result = random.choice(['Y', 'N', 'A'])
        invalid_responses_gemini.append(i)
        pred_gemini.append(result)

    time.sleep(5) # Throttling/Backoff to avoid "429 RESOURCE_EXHAUSTED"!

print(f"\nTotal invalid responses: {len(invalid_responses_gemini)}")

30it [03:18,  6.60s/it]


Total invalid responses: 0


In [53]:
calculate_metrics(pred_gemini, votes)

Accuracy: 0.9
AUC: nan
Balanced Accuracy: 0.48214285714285715
Precision: 0.3333333333333333
Recall: 0.32142857142857145
F1: 0.32727272727272727
PR AUC: 0.3547619047619048
MCC: 0.4008918628686366
G-Mean: 0.5574175147179049
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean
0.9000 nan 0.4821 0.3333 0.3214 0.3273 0.3548 0.4009 0.5574


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-d

## Gemini-2.5-flash-lite

Modify the model to Gemini 2.5 Flash lite in Vertex API

In [54]:
llm2 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    api_key=VERTEX_API_KEY,
    vertexai=True,
    temperature=0.0,
    max_tokens=5
)

In [55]:
pred_gemini_2 = []
invalid_responses_gemini_2 = []

for i, (draft, country) in tqdm(enumerate(zip(drafts, countries))):

    system_content = "You are an experienced diplomat participating in United Nations Security Council sessions. Your task is to read draft resolutions and vote accordingly. You must strictly respond with 'Y', 'N', or 'A' without any additional explanation."

    user_content = f"""The following is a United Nations Security Council draft resolution. Assume you are a diplomat from {country} and must cast your vote.

    Please read the draft and vote using one of the following options:
    - 'Y' for Yes (in favor)
    - 'N' for No (against)
    - 'A' for Abstain

    Draft Resolution: {draft}

    Now, please output your final vote (Y, N, or A) based on your diplomatic stance:"""

    messages = [
        SystemMessage(content=system_content),
        HumanMessage(content=user_content)
    ]

    try:
        response = llm2.invoke(messages)
        raw_result = response.content.strip()

        result = ""
        for char in raw_result.upper():
            if char in ['Y', 'N', 'A']:
                result = char
                break

        valid_votes = ['Y', 'N', 'A']
        if result not in valid_votes:
            print(f"\n[Debug] Invalid response at index {i}: '{raw_result}'")
            result = random.choice(valid_votes)
            invalid_responses_gemini_2.append(i)

        pred_gemini_2.append(result)

    except Exception as e:
        print(f"\n[Error] API call failed at index {i}. Reason: {e}")
        result = random.choice(['Y', 'N', 'A'])
        invalid_responses_gemini_2.append(i)
        pred_gemini_2.append(result)

    time.sleep(8) # Throttling/Backoff to avoid "429 RESOURCE_EXHAUSTED"

print(f"\nTotal invalid responses: {len(invalid_responses_gemini_2)}")

30it [04:45,  9.52s/it]


Total invalid responses: 0


In [56]:
calculate_metrics(pred_gemini_2, votes)

Accuracy: 0.9666666666666667
AUC: nan
Balanced Accuracy: 0.75
Precision: 0.6666666666666666
Recall: 0.5
F1: 0.5555555555555555
PR AUC: 0.5111111111111111
MCC: 0.7433919416750281
G-Mean: 0.7031674369909662
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean
0.9667 nan 0.7500 0.6667 0.5000 0.5556 0.5111 0.7434 0.7032


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
